In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

fact = pd.read_parquet("data/interim/fact_validations.parquet")
os.makedirs("data/processed", exist_ok=True)

print(fact.shape)
print(fact.columns.tolist())

(67407, 37)
['validation_id', 'validation_ts', 'route_id', 'vehicle_id', 'stop_id', 'card_id', 'fare_type', 'fare_amount', 'passenger_count', 'transfer_flag', 'is_reversed', 'date', 'year', 'month', 'year_month', 'quarter', 'hour', 'day_of_week', 'is_weekend', 'time_period', 'revenue', 'route_name', 'mode', 'route_district', 'route_length_km', 'scheduled_trips_per_day', 'vehicle_type', 'capacity', 'year_manufactured', 'depot', 'year_manufactured ', 'vehicle_age_band', 'stop_name', 'district_y', 'zone', 'has_shelter', 'load_factor']


#### 1. Table agg_ridership_by_month

In [2]:
agg_ridership_by_month=fact.groupby("year_month").agg(
    total_boardings=("validation_id", "count"),
    total_revenue=("revenue","sum"),
    unique_cards=("card_id", "nunique")
).reset_index()

agg_ridership_by_month["total_revenue"]= agg_ridership_by_month["total_revenue"].round(2)
agg_ridership_by_month= agg_ridership_by_month.sort_values("year_month")

print(agg_ridership_by_month.head())

  year_month  total_boardings  total_revenue  unique_cards
0    2024-01            10042       19868.69          7429
1    2024-02             9083       17989.23          6891
2    2024-03             9944       19682.43          7417
3    2024-04             9505       18865.63          7133
4    2024-05             9974       19561.95          7416


In [3]:
days_per_month= fact.groupby("year_month")["date"].nunique().reset_index(name="days_in_period")
agg_ridership_by_month= agg_ridership_by_month.merge(days_per_month, on="year_month")
agg_ridership_by_month["avg_daily_boarding"]=(
    agg_ridership_by_month["total_boardings"]/ agg_ridership_by_month["days_in_period"]
).round(2)
agg_ridership_by_month= agg_ridership_by_month.drop(columns="days_in_period")

print(agg_ridership_by_month)

   year_month  total_boardings  total_revenue  unique_cards  \
0     2024-01            10042       19868.69          7429   
1     2024-02             9083       17989.23          6891   
2     2024-03             9944       19682.43          7417   
3     2024-04             9505       18865.63          7133   
4     2024-05             9974       19561.95          7416   
5     2024-06             9498       18774.68          7145   
6     2024-07             1543        3097.00          1345   
7     2024-08             1491        2988.20          1292   
8     2024-09             1636        3233.30          1410   
9     2024-10             1607        3182.73          1365   
10    2024-11             1524        3022.91          1300   
11    2024-12             1560        3056.07          1319   

    avg_daily_boarding  
0               323.94  
1               313.21  
2               320.77  
3               316.83  
4               321.74  
5               316.60  
6    

In [4]:
assert agg_ridership_by_month["year_month"].isna().sum()== 0, "Nulls found in year_month"

In [5]:
agg_ridership_by_month.to_csv("data/processed/agg_ridership_by_month.csv", index=False)

#### 2. Table agg_ridership_by_hour

In [6]:
fact["day_type"]= fact["is_weekend"].map({True: "Weekend", False:"Weekday"})


In [7]:
agg_ridership_by_hour= fact.groupby(["hour", "day_type"]).agg(
    total_boardings= ("validation_id", "count")
).reset_index()
print(agg_ridership_by_hour.head())

   hour day_type  total_boardings
0     0  Weekday              111
1     0  Weekend               46
2     1  Weekday              100
3     1  Weekend               33
4     2  Weekday              120


In [8]:
days_per_type= fact.groupby("day_type")["date"].nunique().reset_index(name="days_in_period")
agg_ridership_by_hour= agg_ridership_by_hour.merge(days_per_type, on= "day_type")

agg_ridership_by_hour["avg_boarding_per_day"]= (
    agg_ridership_by_hour["total_boardings"] / agg_ridership_by_hour["days_in_period"]
).round(2)

agg_ridership_by_hour= agg_ridership_by_hour.drop(columns="days_in_period")

In [9]:
agg_ridership_by_hour["pct_of_daily_total"]= ( 
    agg_ridership_by_hour["total_boardings"]
    / agg_ridership_by_hour.groupby("day_type")["total_boardings"].transform("sum") *100
).round(2)
agg_ridership_by_hour= agg_ridership_by_hour.sort_values(["day_type", "hour"])
print(agg_ridership_by_hour.head(10))

    hour day_type  total_boardings  avg_boarding_per_day  pct_of_daily_total
0      0  Weekday              111                  0.71                0.23
2      1  Weekday              100                  0.64                0.21
4      2  Weekday              120                  0.76                0.25
6      3  Weekday              103                  0.66                0.21
8      4  Weekday              143                  0.91                0.30
10     5  Weekday              299                  1.90                0.62
12     6  Weekday             1254                  7.99                2.59
14     7  Weekday             3850                 24.52                7.96
16     8  Weekday             5887                 37.50               12.17
18     9  Weekday             4401                 28.03                9.10


In [10]:
assert agg_ridership_by_hour["hour"].isna().sum()==0
assert agg_ridership_by_hour["day_type"].isna().sum()==0

agg_ridership_by_hour.to_csv("data/processed/agg_ridership_by_hour.csv", index=False)

In [11]:
print(agg_ridership_by_hour.groupby("day_type")["pct_of_daily_total"].sum())

day_type
Weekday    100.02
Weekend     99.97
Name: pct_of_daily_total, dtype: float64


#### 3. Table agg_route_performance


In [12]:
agg_route_performance = fact.groupby(
    ["route_id", "route_name", "mode", "route_district", "year_month"]
).agg(
    total_boardings= ("validation_id", "count"),
    total_revenue= ("revenue", "sum"),
    route_length_km= ("route_length_km", "first"),
    avg_load_factor= ("load_factor", "mean")
).reset_index()

In [13]:
agg_route_performance["boarding_per_km"]= (
    agg_route_performance["total_boardings"] / agg_route_performance["route_length_km"]
).round(2)
agg_route_performance["total_revenue"]= agg_route_performance["total_revenue"].round(2)
agg_route_performance["avg_load_factor"]= agg_route_performance["avg_load_factor"].round(4)

agg_route_performance= agg_route_performance.drop(columns= "route_length_km")

In [14]:
agg_route_performance = agg_route_performance.sort_values(["year_month", "route_id"])

assert agg_route_performance["route_id"].isna().sum()== 0
assert agg_route_performance["year_month"].isna().sum()==0

agg_route_performance.to_csv("data/processed/agg_route_preformance.csv", index=False)
print(agg_route_performance.head())

   route_id            route_name   mode route_district year_month  \
0      R001     Angel Harbor Line   Tram    Harbourside    2024-01   
12     R002   Jeffrey Street Line    Bus        Unknown    2024-01   
24     R003     Erica Plains Line  Metro      Southbank    2024-01   
36     R004  Brittany Bypass Line   Tram      Northgate    2024-01   
48     R005     Clayton Fort Line    Bus        Unknown    2024-01   

    total_boardings  total_revenue  avg_load_factor  boarding_per_km  
0               184         346.30           0.0053             9.29  
12              157         300.29           0.0131            12.46  
24              162         316.07           0.0018            37.67  
36              145         285.99           0.0052             8.24  
48              161         306.07           0.0140            32.20  


#### 4. Table agg_mode_summary

In [15]:
agg_mode_summary=fact.groupby(["mode", "year_month"]).agg(
    total_boardings= ("validation_id", "count"),
    total_revenue= ("revenue", "sum"),
    route_count=("route_id", "nunique")
).reset_index()



In [16]:
agg_mode_summary["boardings_per_route"]=(
    agg_mode_summary["total_boardings"] /
    agg_mode_summary["route_count"]
)

agg_mode_summary["total_revenue"]= (
    agg_mode_summary["total_revenue"].round(2)
)
agg_mode_summary["boardings_per_route"]=(
    agg_mode_summary["boardings_per_route"].round(4)
)

In [17]:
agg_mode_summary=agg_mode_summary.sort_values(["year_month", "mode"]).reset_index(drop=True)

assert agg_mode_summary[["mode", "year_month"]].notna().all().all()

agg_mode_summary.head()

,mode,year_month,total_boardings,total_revenue,route_count,boardings_per_route
0,Bus,2024-01,5772,11462.98,34,169.7647
1,Metro,2024-01,959,1914.27,6,159.8333
2,Tram,2024-01,3311,6491.44,20,165.5500
3,Bus,2024-02,5092,10066.76,34,149.7647
4,Metro,2024-02,872,1744.67,6,145.3333


#### 5. Table agg_fare_type_mix

In [18]:
agg_fare_type_mix= (
    fact.groupby(["fare_type", "year_month"], as_index=False).agg(
        boardings= ("validation_id","count"),
        revenue= ("fare_amount", "sum"),
    )
)
agg_fare_type_mix["pct_of_monthly_boardings"]=(
    agg_fare_type_mix["boardings"]/
    agg_fare_type_mix.groupby("year_month")["boardings"].transform("sum")
).round(4)

agg_fare_type_mix["revenue"]= agg_fare_type_mix["revenue"].round(2)

agg_fare_type_mix= agg_fare_type_mix.sort_values(["year_month", "fare_type"]).reset_index(drop=True)

assert agg_fare_type_mix[["fare_type", "year_month"]].notna().all().all()
agg_fare_type_mix.head()

,fare_type,year_month,boardings,revenue,pct_of_monthly_boardings
0,Adult,2024-01,5572,13925.96,0.5549
1,Child,2024-01,819,815.94,0.0816
2,Pass Holder,2024-01,1014,60.56,0.1010
3,Senior,2024-01,1172,1404.67,0.1167
4,Student,2024-01,1465,2200.10,0.1459


In [19]:
agg_fare_type_mix.groupby("year_month")["pct_of_monthly_boardings"].sum()

year_month
2024-01    1.0001
2024-02    1.0000
2024-03    1.0001
2024-04    1.0000
2024-05    1.0000
2024-06    0.9999
2024-07    1.0000
2024-08    1.0000
2024-09    1.0001
2024-10    1.0000
2024-11    0.9999
2024-12    1.0000
Name: pct_of_monthly_boardings, dtype: float64

#### 6. Table agg_stop_activity

In [20]:
agg_stop_activity= (
    fact.groupby(["stop_id", "stop_name", "district_y", "zone"], as_index= False)
    .agg(
        total_boardings=("validation_id", "count"),
        has_shelter=("has_shelter", "first"),
    )
)

agg_stop_activity["rank_within_district"]=(
    agg_stop_activity.groupby("district_y")["total_boardings"].rank(method="min", ascending=False)
    .astype(int)
)

agg_stop_activity=agg_stop_activity.sort_values(
    ["district_y", "rank_within_district"]
).reset_index(drop=True)

assert agg_stop_activity[["stop_id", "stop_name","district_y", "zone"]].notna().all().all()
agg_stop_activity.head()


,stop_id,stop_name,district_y,zone,total_boardings,has_shelter,rank_within_district
0,S0394,Tucker Mills Stop,Centro,Zone 2,183,False,1
1,S0037,Emily Hills Stop,Centro,Zone 1,182,True,2
2,S0103,Jennifer Lights Stop,Centro,Zone 1,181,False,3
3,S0047,Joseph Coves Stop,Centro,Zone 3,179,False,4
4,S0186,James Streets Stop,Centro,Zone 2,179,False,4


#### 7.Table agg_vehicle_utilisation

In [21]:
agg_vehicle_utilisation= (
    fact.groupby(["vehicle_id", "vehicle_type","depot", "vehicle_age_band"], as_index=False)
    .agg(
        total_boardings=("validation_id", "count"),
        avg_load_factor=("load_factor", "mean"),
        active_days= ("date", "nunique"),
    )
)

agg_vehicle_utilisation["avg_load_factor"]= agg_vehicle_utilisation["avg_load_factor"].round(4)

agg_vehicle_utilisation= agg_vehicle_utilisation.sort_values(
    "total_boardings", ascending=False
).reset_index(drop=True)

assert agg_vehicle_utilisation[
    ["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"]
].notna().all().all()
agg_vehicle_utilisation.head()

,vehicle_id,vehicle_type,depot,vehicle_age_band,total_boardings,avg_load_factor,active_days
0,V0176,Tram,North Depot,20+,683,0.0061,203
1,V0116,Tram,Riverside Depot,11-20,640,0.0058,196
2,V0040,Tram,Central Depot,20+,629,0.0058,196
3,V0177,Tram,East Depot,11-20,626,0.0047,200
4,V0139,Tram,North Depot,0-5,625,0.0057,201


### 8. Table agg_peak_analysis

In [22]:
agg_peak_analysis=(
    fact.groupby(["time_period", "mode"], as_index=False)
    .agg(
        total_boardings=("validation_id", "count"),
        avg_load_factor= ("load_factor", "mean"),
    )
)

agg_peak_analysis["pct_of_mode_total"]=(
    agg_peak_analysis["total_boardings"]/ agg_peak_analysis.groupby("mode")["total_boardings"].transform("sum")
).round(4)

agg_peak_analysis["avg_load_factor"]= agg_peak_analysis["avg_load_factor"].round(4)

agg_peak_analysis= agg_peak_analysis.sort_values(
    ["mode", "time_period"]
).reset_index(drop=True)

assert agg_peak_analysis[["time_period", "mode"]].notna().all().all()
agg_peak_analysis

,time_period,mode,total_boardings,avg_load_factor,pct_of_mode_total
0,AM Peak,Bus,11158,0.0134,0.2923
1,Early,Bus,1301,0.0133,0.0341
2,Evening,Bus,3787,0.0135,0.0992
3,Midday,Bus,9424,0.0135,0.2469
4,Night,Bus,355,0.0126,0.0093
5,PM Peak,Bus,12152,0.0134,0.3183
6,AM Peak,Metro,1904,0.0018,0.2925
7,Early,Metro,216,0.0018,0.0332
8,Evening,Metro,649,0.0018,0.0997
9,Midday,Metro,1622,0.0018,0.2492


In [23]:
agg_peak_analysis.groupby("mode")["pct_of_mode_total"].sum()

mode
Bus      1.0001
Metro    1.0000
Tram     0.9999
Name: pct_of_mode_total, dtype: float64

#### 9. Table agg_transfer_behaviour

In [24]:
agg_transfer_behaviour= (
    fact.groupby(["year_month", "mode"], as_index=False)
    .agg(
        transfer_count=("transfer_flag", "sum"),
        transfer_rate= ("transfer_flag", "mean"),
    )
)

agg_transfer_behaviour["transfer_count"]= agg_transfer_behaviour["transfer_count"].astype(int)
agg_transfer_behaviour["transfer_rate"]= agg_transfer_behaviour["transfer_rate"].round(4)

agg_transfer_behaviour= agg_transfer_behaviour.sort_values(
    ["year_month","mode"]
).reset_index(drop= True)

assert agg_transfer_behaviour[["year_month", "mode"]].notna().all().all()
agg_transfer_behaviour.head()

,year_month,mode,transfer_count,transfer_rate
0,2024-01,Bus,2160,0.3742
1,2024-01,Metro,365,0.3806
2,2024-01,Tram,1263,0.3815
3,2024-02,Bus,1882,0.3696
4,2024-02,Metro,297,0.3406


In [25]:
from pathlib import Path
from datetime import datetime

out_dir= Path("..data/output")
out_dir.mkdir(parents=True, exist_ok=True)

tables= {
    "agg_ridership_by_month":agg_ridership_by_month,
    "agg_ridership_by_hour":agg_ridership_by_hour,
    "agg_route_performance":agg_route_performance,
    "agg_mode_summary": agg_mode_summary,
    "agg_fare_type_mix":agg_fare_type_mix,
    "agg_stop_activity":agg_stop_activity,
    "agg_vehicle_utilisation":agg_vehicle_utilisation,
    "agg_peak_analysis":agg_peak_analysis,
    "agg_transfer_behaviour":agg_transfer_behaviour
}

keys= {
     "agg_ridership_by_month":["year_month"],
        "agg_ridership_by_hour":["hour", "day_type"],
        "agg_route_performance":["route_id",  "route_name", "mode", "route_district", "year_month"],
        "agg_mode_summary": ["mode", "year_month"],
        "agg_fare_type_mix":["fare_type", "year_month"],
        "agg_stop_activity":[ "stop_id","stop_name", "district_y", "zone"],
        "agg_vehicle_utilisation":["vehicle_id", "vehicle_type", "depot", "vehicle_age_band"],
        "agg_peak_analysis":["time_period", "mode"],
        "agg_transfer_behaviour":["year_month", "mode"],

}

manifest_rows=[]
ts = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

for name, df_out in tables.items():
    assert df_out[keys[name]].notna().all().all(), f"Nulls in key columns of {name}"
    path = out_dir / f"{name}.csv"
    df_out.to_csv(path, index=False, encoding="utf-8")
    manifest_rows.append({
        "file": path.name,
        "rows": len(df_out),
        "columns": df_out.shape[1],
        "generated_at": ts
    })

    manifest= pd.DataFrame(manifest_rows)
    manifest.to_csv(out_dir / "_manifest.csv", index=False, encoding="utf-8")
    manifest